In [0]:
import os, shutil

SRC = os.path.abspath("../data")             # the repo's data folder, found relative to this notebook
DST = "/Volumes/workspace/iaa/raw/week1"     # where the copies will live
SCHEMA = "workspace.iaa"

files = sorted(f for f in os.listdir(SRC) if f.endswith(".parquet"))
print(SRC)
print(len(files), "files:", files)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.iaa.raw COMMENT 'Kaggle Week 1 files, copied from the Git repo'")
os.makedirs(DST, exist_ok=True)

for f in files:
    shutil.copyfile(os.path.join(SRC, f), os.path.join(DST, f))
print(os.listdir(DST))

In [0]:
for f in files:
    name = f.removesuffix(".parquet")
    (spark.read.parquet(f"{DST}/{f}")
          .write.mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(f"{SCHEMA}.{name}"))
    print(f"{name:20s} {spark.table(f'{SCHEMA}.{name}').count():>10,} rows")

In [0]:
problems = []
for f in files:
    name = f.removesuffix(".parquet")
    src = [(c.name, c.dataType.simpleString()) for c in spark.read.parquet(f"{DST}/{f}").schema]
    tbl = [(c.name, c.dataType.simpleString()) for c in spark.table(f"{SCHEMA}.{name}").schema]
    if src != tbl:
        problems.append(f"{name}: table columns/types differ from the source file")
    for col, typ in tbl:
        if col.endswith("_id") and col != "branch_id" and typ != "string":
            problems.append(f"{name}.{col} is {typ}, expected string")
        if col in ("period_start", "period_end", "loss_date", "report_date", "date_of_birth") and typ != "date":
            problems.append(f"{name}.{col} is {typ}, expected date")

assert not problems, "\n".join(problems)
print(f"All {len(files)} tables: types match the source files, IDs are text, dates are dates")